# S6_3 — Attention Map Visualisation

**Leaf-JEPA IRP** | Stage 6 — Analysis and Interpretation


**Purpose:** Compare WHERE models focus: generic I-JEPA vs Leaf-JEPA vs best PEFT.
If domain pretraining works, Leaf-JEPA attention should concentrate on disease lesions.

**Outputs:**
- `attention_comparison_grid.png` — flagship qualitative figure
- `attention_iou_disease_region.csv` — quantitative IoU with disease saliency
- Individual per-image attention maps in `figures/attention_individual/`

**Research Questions Served:** RQ1 (visual evidence), RQ4 (domain-specific focus)


## Initialization

In [1]:
import sys, json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import OrderedDict
import torch
from PIL import Image

PROJECT_ROOT = Path("..").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from stage6_analysis_and_interpretation.config_stage6 import *
from stage6_analysis_and_interpretation.analysis_utils import (
    set_seed, get_device, load_json, save_json,
    load_ijepa_encoder, SimpleImageDataset, get_eval_transform,
    load_split_data, extract_attention_maps, attention_to_heatmap,
    compute_attention_iou, plot_attention_comparison, print_section,
)
import matplotlib.pyplot as plt

set_seed(RANDOM_SEED)
device = get_device()
STAGE6_FIGURES.mkdir(parents=True, exist_ok=True)
STAGE6_DATA.mkdir(parents=True, exist_ok=True)
(STAGE6_FIGURES / "attention_individual").mkdir(exist_ok=True)


## Select sample Images

In [ ]:
print_section("Selecting Sample Images")

image_paths, labels, class_names = load_split_data(SPLITS_DIR, split="test")

# Select ATTN_SAMPLES_PER_CATEGORY images per pathogen category
# Build pathogen mapping
pathogen_map = {}  # class_name -> category
for cat, classes in PATHOGEN_CATEGORIES.items():
    for cls in classes:
        pathogen_map[cls] = cat

# If PATHOGEN_CATEGORIES is not filled, use a simple heuristic
if not any(PATHOGEN_CATEGORIES.values()):
    print("  \u26a0\ufe0f PATHOGEN_CATEGORIES not filled in config.")
    print("  Selecting samples uniformly across classes instead.")
    # Select from diverse classes
    rng = np.random.RandomState(RANDOM_SEED)
    selected_indices = []
    unique_labels = np.unique(labels)
    # Pick from first 5 classes and last 5 classes for diversity
    sample_classes = list(unique_labels[:5]) + list(unique_labels[-5:])
    for cls in sample_classes[:ATTN_SAMPLES_PER_CATEGORY * 5]:
        cls_indices = [i for i, l in enumerate(labels) if l == cls]
        if cls_indices:
            selected_indices.append(rng.choice(cls_indices))
else:
    selected_indices = []
    rng = np.random.RandomState(RANDOM_SEED)
    for category in ["Fungal", "Bacterial", "Viral", "Abiotic", "Healthy"]:
        cat_classes = PATHOGEN_CATEGORIES.get(category, [])
        cat_indices = []
        for cls_name in cat_classes:
            if cls_name in class_names:
                cls_idx = class_names.index(cls_name)
                cls_sample_indices = [i for i, l in enumerate(labels) if l == cls_idx]
                cat_indices.extend(cls_sample_indices)
        if cat_indices:
            n_select = min(ATTN_SAMPLES_PER_CATEGORY, len(cat_indices))
            selected_indices.extend(rng.choice(cat_indices, n_select, replace=False))

print(f"  Selected {len(selected_indices)} images for attention visualisation")

# Load selected images
sample_paths = [image_paths[i] for i in selected_indices]
sample_labels = [labels[i] for i in selected_indices]
sample_images_pil = [Image.open(p).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE)) for p in sample_paths]

for i, (p, l) in enumerate(zip(sample_paths, sample_labels)):
    print(f"    [{i}] {class_names[l]}: {Path(p).name}")


## Load (Encoders and Extract Attention)

In [ ]:
print_section("Extracting Attention Maps")

transform = get_eval_transform(NORM_MEAN, NORM_STD, IMAGE_SIZE)
sample_tensors = torch.stack([transform(img) for img in sample_images_pil])

attn_maps_dict = OrderedDict()

# Generic I-JEPA
if IJEPA_CHECKPOINT.exists():
    print("  Generic I-JEPA...")
    model = load_ijepa_encoder(IJEPA_CHECKPOINT, device=device)
    attn = extract_attention_maps(model, sample_tensors, device=device)
    attn_maps_dict["Generic I-JEPA"] = attn
    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

# Leaf-JEPA
if LEAF_JEPA_CHECKPOINT.exists():
    print("  Leaf-JEPA...")
    model = load_ijepa_encoder(LEAF_JEPA_CHECKPOINT, device=device)
    attn = extract_attention_maps(model, sample_tensors, device=device)
    attn_maps_dict["Leaf-JEPA"] = attn
    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

print(f"  Attention extracted for: {list(attn_maps_dict.keys())}")


## Plot Attention Comparison grid

In [ ]:
print_section("Attention Comparison Grid")

plot_attention_comparison(
    sample_images_pil,
    attn_maps_dict,
    STAGE6_FIGURES / "attention_comparison_grid.png",
    grid_size=GRID_SIZE,
    image_size=IMAGE_SIZE,
    alpha=ATTN_ALPHA,
    cmap=ATTN_COLORMAP,
)

# Save individual attention maps
for model_name, attn_arr in attn_maps_dict.items():
    for i in range(len(sample_images_pil)):
        heatmap = attention_to_heatmap(attn_arr[i], GRID_SIZE, IMAGE_SIZE)
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.imshow(np.array(sample_images_pil[i]))
        ax.imshow(heatmap, cmap=ATTN_COLORMAP, alpha=ATTN_ALPHA)
        ax.set_title(f"{model_name}: {class_names[sample_labels[i]]}", fontsize=10)
        ax.axis("off")
        safe_name = model_name.replace(" ", "_").replace("-", "_")
        fig.savefig(STAGE6_FIGURES / "attention_individual" / f"{safe_name}_img{i}.png")
        plt.close(fig)

print("  Individual maps saved.")


## Attention IoU with Disease saliency

In [ ]:
print_section("Attention IoU Analysis")

# Compute disease-region saliency for each sample image
# using hue deviation from healthy leaf (same approach as Stage 4)

iou_results = []

for i, img_pil in enumerate(sample_images_pil):
    img_np = np.array(img_pil).astype(np.float32) / 255.0
    # Convert to HSV for saliency
    import colorsys
    h_channel = np.zeros((IMAGE_SIZE, IMAGE_SIZE))
    s_channel = np.zeros((IMAGE_SIZE, IMAGE_SIZE))
    for r in range(IMAGE_SIZE):
        for c in range(IMAGE_SIZE):
            h, s, v = colorsys.rgb_to_hsv(img_np[r, c, 0], img_np[r, c, 1], img_np[r, c, 2])
            h_channel[r, c] = h
            s_channel[r, c] = s

    # Per-patch saliency (16x16 grid)
    patch_saliency = np.zeros(NUM_PATCHES)
    for pi in range(GRID_SIZE):
        for pj in range(GRID_SIZE):
            r0, r1 = pi * PATCH_SIZE, (pi + 1) * PATCH_SIZE
            c0, c1 = pj * PATCH_SIZE, (pj + 1) * PATCH_SIZE
            patch_h = h_channel[r0:r1, c0:c1].mean()
            patch_s = s_channel[r0:r1, c0:c1].mean()
            # Hue deviation from typical healthy green (~0.25-0.35 in [0,1] hue)
            healthy_hue = 0.30
            hue_dev = min(abs(patch_h - healthy_hue), 1 - abs(patch_h - healthy_hue))
            # Weight by saturation (ignore grey/background patches)
            patch_saliency[pi * GRID_SIZE + pj] = hue_dev * patch_s

    # Compute IoU for each model
    row = {"image_idx": i, "class": class_names[sample_labels[i]]}
    for model_name, attn_arr in attn_maps_dict.items():
        iou = compute_attention_iou(attn_arr[i], patch_saliency, top_k_pct=ATTN_TOP_K_PCT)
        row[f"iou_{model_name.replace(' ', '_')}"] = iou
    iou_results.append(row)

iou_df = pd.DataFrame(iou_results)
iou_df.to_csv(STAGE6_DATA / "attention_iou_disease_region.csv", index=False)
print(iou_df.to_string(index=False))

# Summary
print("\nMean IoU per model:")
for col in iou_df.columns:
    if col.startswith("iou_"):
        model = col.replace("iou_", "")
        print(f"  {model}: {iou_df[col].mean():.4f}")

print("\n\u2705 S6_3 complete.")


## Critical Analysis

### Interpreting Attention Maps
- Higher IoU between Leaf-JEPA attention and disease saliency = model focuses on disease regions
- If generic I-JEPA focuses on leaf edges / background while Leaf-JEPA focuses on lesions, this is
  the strongest qualitative evidence for RQ1
- Include ALL sampled images, not just the ones that look good — cherry-picking undermines credibility

### Limitations
- Self-attention from the last block is one interpretation of "where the model looks"
- Alternative: Grad-CAM requires a classification head (not available for frozen encoders)
- The IoU metric depends on the saliency map quality — use the same healthy-hue parameters from Stage 4

### Dissertation Figure
The attention comparison grid is a Figure in Chapter 4 (Results). Caption should state:
"Attention maps from the final transformer block, averaged across all attention heads.
Warmer colours indicate higher attention. Generic I-JEPA (left) and Leaf-JEPA (right)
show [describe observed difference]."
